# Greybound WAV comparison dashboard

Notebook visuel pour inspecter une candidate Greybound contre une reference: alignement, niveaux, spectre, residuel, enveloppe et segments.

## Inputs

Modifier ces chemins, puis executer les cellules dans l'ordre. Depuis la racine du repo, lancer avec:

```sh
uv --project lab run --with jupyter jupyter lab lab/notebooks/wav-comparison-dashboard.ipynb
```

In [ ]:
from pathlib import Path

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parents[1]
elif REPO.name == "lab":
    REPO = REPO.parent

candidate_wav = REPO / "lab/renders/nox30-driven.wav"
reference_wav = REPO / "lab/references/nox30-reference.wav"
segments_json = REPO / "lab/segments/guitar-chords.markers.json"

segments_path = segments_json if segments_json.exists() else None
candidate_wav, reference_wav, segments_path

In [ ]:
import matplotlib.pyplot as plt

from greybound_lab.notebook import (
    load_wav_comparison,
    plot_diagnostic_bars,
    plot_overview,
    plot_residual_spectrogram,
    plot_segment_band_residuals,
    plot_spectrum,
    print_metric_summary,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

## Load and summarize

In [ ]:
analysis = load_wav_comparison(candidate_wav, reference_wav, segments_path=segments_path)
print_metric_summary(analysis)

## Global diagnostics

In [ ]:
plot_overview(analysis, start_s=0.0, end_s=min(analysis.duration_seconds, 4.0));

In [ ]:
plot_spectrum(analysis);

In [ ]:
plot_diagnostic_bars(analysis);

In [ ]:
plot_residual_spectrogram(analysis);

## Segment diagnostics

Si un fichier de marqueurs est fourni, cette vue montre ou le residuel vit par bande musicale.

In [ ]:
segment_figure = plot_segment_band_residuals(analysis)
segment_figure

In [ ]:
for segment in analysis.metrics.segments:
    print(
        f"{segment.name:24s} "
        f"null={segment.null_relative_db:7.2f} dB  "
        f"lsd={segment.log_spectral_distance_db:6.2f} dB  "
        f"env={segment.envelope_error_db:7.2f} dB"
    )

## Zoom window

Changer `zoom_start_s` / `zoom_end_s` pour inspecter un accord, une attaque ou une zone suspecte du spectrogramme residuel.

In [ ]:
zoom_start_s = 0.0
zoom_end_s = min(analysis.duration_seconds, 0.5)

plot_overview(analysis, start_s=zoom_start_s, end_s=zoom_end_s);